## <font color='red'> INSTRUCTIONS </font>

<b> 
1. Write your code only in cells below the "WRITE CODE BELOW" title. Do not modify the code below the "DO NOT MODIFY" title. <br>
2. The expected data types of the output answers for each question are given in the last cell through assertion statements. Your answers must match these expected output data types. Hint: Many of the answers need to be a Python dictionary. Consider methods like to_dict() to convert a Pandas Series to a dictionary. <br>
3. The answers are then written to a JSON file named my_results_PA1.json. You can compare this with the provided expected output file "expected_results_PA1.json". <br>
4. After you complete writing your code, click "Kernel -> Restart Kernel and Run All Cells" on the top toolbar. There should NOT be any syntax/runtime errors, otherwise points will be deducted. <br>
5. For submitting your solution, first download your notebook by clicking "File -> Download". Rename the file as &ltTEAM_ID&gt.ipynb" and upload to Canvas.</b>


## <font color='red'> DO NOT MODIFY </font>

In [28]:
import time
import json
import dask
import dask.dataframe as dd
import pandas as pd
import ast
import re
from dask.distributed import Client
import ctypes
import numpy as np

def trim_memory() -> int:
    """
    helps to fix any memory leaks.
    """
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client = Client("127.0.0.1:8786")
client.run(trim_memory)
client = client.restart()
print(client)

OSError: Timed out trying to connect to tcp://127.0.0.1:8786 after 30 s

In [ ]:
start = time.time()

## <font color='blue'> WRITE CODE BELOW </font>

In [24]:
user_reviews = dd.read_csv('user_reviews.csv').repartition(npartitions=10)
products = dd.read_csv('products.csv', dtype={'asin': 'object'}).repartition(npartitions=10)

In [3]:
num_reviews = user_reviews.shape[0]
num_products = products.shape[0]

In [ ]:
## Question 1
# num_nulls = user_reviews.map_partitions(lambda x: x.isna().sum()).compute()
start = time.time()
num_nulls = user_reviews.isna().sum().compute()
num_nulls = np.round((num_nulls / user_reviews.shape[0]) * 100, 2)
ans1 = num_nulls.to_dict()
end = time.time()
ans1

In [ ]:
end - start

In [ ]:
## Question 2
start = time.time()
ans2 = products.isna().sum().compute()
ans2 = np.round((ans2 / num_products) * 100, 2)
ans1 = ans2.to_dict()
end = time.time()
ans2

In [ ]:
end - start

In [27]:
## Question 3
start = time.time()
price = products[['price']].reset_index(drop=True)
ratings = user_reviews[['overall']].reset_index(drop=True)
joined = dd.concat([price, ratings], axis=0)
correlation = joined.corr().compute()
ans3 = correlation.loc['price','overall']
end = time.time()
ans3

FutureCancelledError: ('corr-tree-f775378a554c768c191d96ca251d7232', 0) cancelled for reason: scheduler-connection-lost.
Client lost the connection to the scheduler. Please check your connection and re-run your work.

In [21]:
start = time.time()
price = products['price'].dropna().reset_index(drop=True)
ratings = user_reviews['overall'].dropna().reset_index(drop=True)
correlation = price.corr(ratings).compute()
correlation
end = time.time()

ValueError: Unable to concatenate DataFrame with unknown division specifying axis=1

In [16]:
start = time.time()

# Drop NaNs and reset index for alignment
price = products[['price']].dropna().reset_index(drop=True)
ratings = user_reviews[['overall']].dropna().reset_index(drop=True)
ratings = ratings.rename(columns={'overall': 'ratings'})

# Combine into one DataFrame
joined = dd.concat([price, ratings], axis=1).dropna()

# Compute correlation matrix
correlation_matrix = joined.corr().compute()

# Extract just the correlation value between 'price' and 'ratings'
correlation = correlation_matrix.loc['price', 'ratings']

end = time.time()

print("Correlation:", correlation)
print("Time elapsed:", round(end - start, 2), "seconds")

ValueError: Unable to concatenate DataFrame with unknown division specifying axis=1

In [ ]:
### read in the 'user_reviews.csv' and 'products.csv' files, perform your calculations and place the answers in variables ans1 - ans7.


# substitute 'None' with the outputs from your calculations. 
# The expected output types can be seen in the assertion statements below
ans1 = ans1
ans2 = ans2
ans3 = None
ans4 = None
ans5 = None
ans6 = None
ans7 = None

## <font color='red'> DO NOT MODIFY </font>

In [ ]:
end = time.time()

In [ ]:
print(f"execution time = {end-start}s")

In [ ]:
# DO NOT MODIFY
assert type(ans1) == dict, f"answer to question 1 must be a dictionary like {{'reviewerID':0.2, ..}}, got type = {type(ans1)}"
assert type(ans2) == dict, f"answer to question 2 must be a dictionary like {{'asin':0.2, ..}}, got type = {type(ans2)}"
assert type(ans3) == float, f"answer to question 3 must be a float like 0.8, got type = {type(ans3)}"
assert type(ans4) == dict, f"answer to question 4 must be a dictionary like {{'mean':0.4,'max':0.6,'median':0.6...}}, got type = {type(ans4)}"
assert type(ans5) == dict, f"answer to question 5 must be a dictionary, got type = {type(ans5)}"         
assert ans6 == 0 or ans6==1, f"answer to question 6 must be 0 or 1, got value = {ans6}" 
assert ans7 == 0 or ans7==1, f"answer to question 7 must be 0 or 1, got value = {ans7}" 

ans_dict = {
    "q1": ans1,
    "q2": ans2,
    "q3": ans3,
    "q4": ans4,
    "q5": ans5,
    "q6": ans6,
    "q7": ans7,
    "runtime": end-start
}
with open('my_results_PA1.json', 'w') as outfile: json.dump(ans_dict, outfile)         